# NB 6.5 &mdash; Més enllà de classificar

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

Fins ara, cada imatge rebia **una sola resposta**: «gat». Però una foto d'un carrer té cotxes, persones i
semàfors, i sovint volem saber quants n'hi ha i on són.

### Què farem avui

Aquesta sessió és per jugar. Primer provarem models ja entrenats, que fan tres feines més:
**detectar** objectes, **retallar-los** píxel a píxel i **trobar les articulacions** d'una persona. Són
CNN per dins, de la família **YOLO** (*You Only Look Once*), i responen en dècimes de segon. Després
n'ensenyarem un a trobar animals que no coneixia, i farem que una CNN pinti com Van Gogh. Totes dues coses van
molt millor amb **GPU**: a Colab, activa-la a *Entorn d'execució → Canvia el tipus d'entorn → GPU T4*. Sense GPU
també funciona, amb un detector que et donem ja entrenat i imatges més petites.
Si et sobra temps, al final hi ha una missió extra amb **CLIP**, un model que tria entre les classes que tu li escrius.

| Feina | Què respon | Model |
|---|---|---|
| Classificar | «Hi ha un gat» | ResNet18, al NB 6.4 |
| **Detectar** | «Hi ha un gat, i és dins d'aquest requadre» | `yolo11n.pt` |
| **Segmentar** | «Hi ha un gat, i aquests píxels són el gat» | `yolo11n-seg.pt` |
| **Poses** | «Aquí hi ha una persona, i aquest és el seu colze» | `yolo11n-pose.pt` |
| **Detectar coses noves** | «Això és un rinoceront», encara que COCO no en tenia | `yolo11n.pt`, reentrenat |
| **Pintar** | «Aquesta és la teva foto, pintada com un quadre» | VGG19 |

### Preguntes que has de saber respondre en acabar

1. Què fa el llindar de confiança, i quin preu té abaixar-lo?
2. Què passa amb allò que no és a les 80 classes que coneix el model?
3. Aquests models aguanten els canvis que vam provar al crash test?
4. Quins problemes planteja detectar i seguir persones?
5. Què li passa al que sabia un detector quan li ensenyes classes noves?
6. Què guarden les primeres capes d'una CNN, i què les profundes?

## 0. Preparació

Aquests models són de la biblioteca **Ultralytics**, amb llicència AGPL-3.0: és de codi obert i, si fas un programa
que la faci servir i el distribueixes o l'ofereixes per internet, n'has de compartir el codi. Per a aquest ús docent, no és cap problema. Cal instal·lar-la.

> ⏱ A Colab tarda menys d'un minut.

In [ ]:
!pip install -q ultralytics

Ara carreguem les fotos de mostra i definim dues funcions per veure els resultats.

In [ ]:
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO

URL_PHOTOS = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT06-Xarxes_convolucionals/fotos/"
PHOTOS = ["carrer_neu.jpg", "dublin_1901.jpg", "cafe.png", "gata.png", "coet.jpg"]
for name in PHOTOS:
    urllib.request.urlretrieve(URL_PHOTOS + name, name)


def show_result(result, size=8):
    """Dibuixa la imatge amb tot el que ha trobat el model."""
    plt.figure(figsize=(size, size))
    plt.imshow(result.plot()[:, :, ::-1])      # result.plot() torna els colors en ordre BGR
    plt.axis("off")
    plt.show()


def count_found(result):
    """Retorna quants objectes de cada classe ha trobat el model."""
    return dict(Counter(result.names[int(number)] for number in result.boxes.cls))

## 1. Detectar

Carreguem el model i li passem una foto. La primera vegada baixa els pesos, uns 6 MB. El paràmetre `conf`
és el **llindar de confiança**: només es dibuixa allò de què n'està prou segur. La funció `predict`
retorna una llista amb un resultat per imatge.

In [ ]:
detector = YOLO("yolo11n.pt")
result = detector.predict("carrer_neu.jpg", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result)

**5 semàfors, 3 persones i 2 cotxes**, cadascun amb el seu requadre i un número: la seva confiança. Aquest model
tarda dècimes de segon. Fixa't també en el que no ha trobat: hi ha més cotxes, al fons. **Per què no els
veu?** Aquí entra el llindar.

### Ara tu · Missió 1: el llindar

Quan una xarxa detecta, proposa molts requadres, i cadascun té una confiança. El llindar decideix quins es queden.

1. **Aposta.** Si abaixes `conf` de 0,25 a 0,05 en aquesta foto, quants cotxes creus que trobarà? I
   si la puges a 0,9?
2. **Experimenta.** Fes una taula amb `count_found` per a `conf` = 0,05, 0,25, 0,5 i 0,9, amb `carrer_neu.jpg` i
   `dublin_1901.jpg`. Dibuixa el resultat de 0,05: quins requadres són errors?
3. **Explica.** Què hi guanyes i què hi perds abaixant el llindar? Quin llindar posaries en un sistema
   que atura un tren si veu una persona a la via? I en un que envia una multa automàtica?

In [ ]:
# El teu codi aquí

## 2. Segmentar i trobar poses

Els altres dos models s'usen exactament igual.

**Segmentar:** a més del requadre, pinta quins píxels formen l'objecte.

In [ ]:
segmenter = YOLO("yolo11n-seg.pt")
result = segmenter.predict("cafe.png", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result, size=6)

**Poses:** busca 17 punts del cos de cada persona (ulls, espatlles, colzes, genolls…) i els uneix. Aquesta foto és
del carrer Westmoreland de Dublín, cap a l'any 1901.

In [ ]:
pose_model = YOLO("yolo11n-pose.pt")
result = pose_model.predict("dublin_1901.jpg", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result)

Ha trobat **8 persones**, en una fotografia de fa més de cent anys. Aquests models s'han entrenat amb fotos
modernes, però un abric, unes espatlles i un cap continuen sent el mateix.

> **Norma de privacitat.** En aquest notebook, les imatges amb persones són només les que et donem. Si vols
> provar fotos pròpies, fes-les només d'objectes. Una detecció de persones és una mena de tractament
> de dades personals, i no es pot fer amb gent que no ho sap.

### Ara tu · Missió 2: què no sap

Aquests models s'han entrenat amb **COCO**, un conjunt amb només 80 classes. Pots veure-les totes amb
`detector.names`.

1. **Aposta.** Mira la llista. Què creus que dirà el detector de `coet.jpg`? I el model de poses, si li ensenyes la
   gata (`gata.png`)?
2. **Experimenta.** Passa les dues fotos pels tres models. Prova també un llindar baix.
3. **Busca** una altra cosa que sigui a la llista i una que no. Si vols, puja una foto d'un objecte amb la icona de
   la carpeta de Colab i passa-la pel detector.
4. **Explica.** Què fa amb una cosa que no coneix: se'n calla, o s'inventa alguna cosa? I el model de poses,
   amb la gata? Quina diferència hi ha amb el que va fer ResNet18 al NB 6.4?

In [ ]:
# El teu codi aquí

## 3. A la paret, una altra vegada

Al NB 6.3 vas veure que una CNN es refia del que ha vist. **Aquests models també?** Amb `Image.open` i els
mètodes de PIL es pot canviar una foto abans de passar-la al model, perquè `predict` també accepta imatges:

```python
image = Image.open("dublin_1901.jpg")
turned = image.rotate(90, expand=True)     # la gira 90 graus
tiny = image.resize((96, 71))              # la fa molt petita
detector.predict(turned, conf=0.25, verbose=False)[0]
```

### Ara tu · Missió 3: el crash test de YOLO

1. **Aposta.** Amb la foto de Dublín girada 90 graus, quantes persones
   creus que trobarà? I cap per avall? I si la redueixes a 96 píxels d'ample?
2. **Experimenta.** Fes una taula amb el nombre de persones per a l'original i per a girs de 45, 90 i 180 graus,
   i per a amples de 480, 240 i 96 píxels, totes amb `conf=0.25`.
3. **Dibuixa** el que passa amb el gir de 180 graus.
4. **Explica.** Com ho arreglaries si la càmera d'un dron ho veu tot des de dalt? Què has après al NB 6.2 que
   hi serviria?

In [ ]:
# El teu codi aquí

## 4. Ensenya-li animals nous

A la missió 2 has vist que el detector no surt de les 80 classes de COCO. COCO té elefants i zebres, però **no té
rinoceronts ni búfals. Ara li ensenyarem a trobar-los.** Farem servir *African Wildlife*, un conjunt d'Ultralytics
amb 1.052 fotos d'entrenament i 225 de validació. A cada foto, algú ha dibuixat a mà un requadre al voltant de cada
búfal, elefant, rinoceront i zebra: és la part més cara de fer un detector propi.

No comencem de zero: partim de `yolo11n.pt`, que ja sap trobar coses, i l'entrenem amb les fotos noves. **És transfer
learning**, com al NB 6.4. I es fa amb una sola línia, `train`, que és el `fit` d'Ultralytics.

#### Un incís: mAP50

Mentre entrena, `train` escriu una taula on la columna important és **mAP50**. És l'encert dels detectors, de 0 a 1:
un requadre compta com a encert si és de la classe correcta i se solapa almenys a la meitat amb el que va dibuixar la
persona. Es calcula per a cada classe i se'n fa la mitjana. **Per a què serveix?** L'encert dels classificadors no val
aquí: un detector pot encertar la classe i posar el requadre a un altre lloc, o trobar-ne dos on n'hi ha un.

> ⏱ Amb GPU, la cel·la entrena 5 epochs i tarda uns minuts. **Sense GPU no surt a compte:** amb imatges més petites i
> 2 epochs ja passa dels 3 minuts, i el detector gairebé no aprèn. Per això, sense GPU, la cel·la carrega un model
> que hem entrenat nosaltres amb 10 epochs, `fauna_yolo11n.pt`. La primera vegada, a més, baixa les fotos: uns 100 MB.

In [ ]:
from pathlib import Path

import torch
from ultralytics.data.utils import check_det_dataset

URL_UT = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT06-Xarxes_convolucionals/"
GPU = torch.cuda.is_available()
wildlife_data = check_det_dataset("african-wildlife.yaml")       # baixa les fotos, si no hi són
if GPU:
    wildlife = YOLO("yolo11n.pt")                                # partim del detector de COCO
    wildlife.train(data="african-wildlife.yaml", epochs=5, imgsz=640, plots=False)
else:
    urllib.request.urlretrieve(URL_UT + "models/fauna_yolo11n.pt", "fauna_yolo11n.pt")
    wildlife = YOLO("fauna_yolo11n.pt")                          # el nostre: 10 epochs amb GPU
print("Classes del detector nou:", wildlife.names)

Ara, la prova: una foto de validació, que el detector nou no ha vist mai, pels dos detectors. Les fotos de validació són
a la carpeta `VAL`.

In [ ]:
VAL = Path(wildlife_data["val"])
rhinos = VAL / "3 (137).jpg"
for label, model in [("detector de COCO", detector), ("detector nou", wildlife)]:
    result = model.predict(rhinos, conf=0.25, verbose=False)[0]
    print(label, count_found(result))
    show_result(result, size=5)

És una mare rinoceront amb la seva cria. El detector de COCO hi veu **2 vaques**: no sap què és un rinoceront i tria
el que més s'hi assembla de la seva llista. El nou hi troba **2 rinoceronts**, amb el model que et donem; si l'has
entrenat tu, el resultat pot canviar una mica.

### Ara tu · Missió 4: el detector de la sabana

1. **Aposta.** Què dirà el detector nou de `carrer_neu.jpg`? Encara sabrà trobar-hi persones i cotxes? I què dirà de
   la gata?
2. **Experimenta.** Passa pels dos detectors `VAL / "1 (120).jpg"`, on hi ha un búfal, i dues fotos més que triïs de
   `VAL` (`sorted(VAL.glob("*.jpg"))` te les llista). Després, passa `carrer_neu.jpg`, `dublin_1901.jpg` i `gata.png`
   pel detector nou. Apunta-ho tot en una taula: foto, detector i què troba.
3. **Si has entrenat amb GPU,** busca a la sortida de `train` el mAP50 de l'última epoch. Quina classe li costa més?
4. **Explica.** Què ha oblidat el detector nou, i per què? Pista: mira `wildlife.names`. Què hauries de fer per tenir
   un detector d'animals i de persones alhora? I què creus que costa més, entrenar-lo o preparar-ne les dades?

In [ ]:
# El teu codi aquí

## 5. Pinta com un quadre

Una CNN entrenada per classificar ha après a separar dues coses. Les capes profundes guarden **què hi ha** a la
imatge: les formes. Les primeres capes, **com està pintada**: colors i textures, que es poden mesurar mirant quins
filtres s'encenen junts, sense importar on.

L'any 2015, Gatys, Ecker i Bethge ho van fer servir per pintar fotos com quadres. Comencen amb la foto i la retoquen
píxel a píxel perquè, dins la xarxa, conservi les formes de la foto i agafi les textures del quadre. **És gradient
descent, però el que s'ajusta no són els pesos, sinó els píxels.** La xarxa és **VGG19** (2014), entrenada amb
ImageNet, i no s'entrena gens.

La funció `stylize(photo, painting, depth=3)` ho fa. El paràmetre `depth` diu de quantes capes es copia l'estil:

| `depth` | D'on es copia l'estil |
|---|---|
| 1 | Només de la primera capa |
| 3 | De la primera capa de cadascun dels tres primers blocs |
| 5 | De tots cinc blocs, fins a les capes més profundes |

`show_style` dibuixa la foto, el quadre i el resultat, i `as_picture` el converteix en una imatge que pot llegir YOLO.

> ⏱ La primera vegada baixa uns 550 MB de pesos de VGG19. Amb GPU, fa 300 passes amb imatges de 384 píxels i tarda
> uns 20 segons. Sense GPU, 120 passes amb 224 píxels: entre 15 i 30 segons a un ordinador normal, i fins a un minut i
> mig al CPU de Colab.

In [ ]:
# Funcions per pintar una foto amb l'estil d'un quadre. No cal modificar-la.
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from torch import nn
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU
SEED = 42

VGG_WEIGHTS = models.VGG19_Weights.IMAGENET1K_V1
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
STYLE_LAYERS = [0, 5, 10, 19, 28]     # la primera convolució de cada un dels 5 blocs de VGG19
CONTENT_LAYER = 21                    # una convolució del bloc 4: formes, no colors
SMOOTHING = 200                       # castiga els píxels veïns molt diferents: menys soroll
_VGG = {}


def _vgg():
    """Carrega VGG19 una sola vegada: la primera crida baixa uns 550 MB de pesos."""
    if not _VGG:
        network = models.vgg19(weights=VGG_WEIGHTS).features[:29].eval()
        for weights in network.parameters():
            weights.requires_grad_(False)
        _VGG["network"] = network
    return _VGG["network"].to(DEVICE)


def default_settings(device):
    """Retorna (mida, passos) per defecte: amb GPU, imatges més grans i més passos."""
    return (384, 300) if device == "cuda" else (224, 120)


def load_picture(path, size):
    """Obre una imatge, la redueix perquè el costat més llarg faci size i la retorna com a tensor (3, alt, ample)."""
    try:
        image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except FileNotFoundError as error:
        raise ValueError(f"No trobo {path}: comprova que l'has baixat o pujat i que el nom és correcte.") from error
    except UnidentifiedImageError as error:
        raise ValueError(f"No es pot obrir {path}. Si és HEIC, d'iPhone, converteix-la a JPG.") from error
    scale = size / max(image.size)
    image = image.resize((round(image.width * scale), round(image.height * scale)), Image.BILINEAR)
    return torch.tensor(np.array(image) / 255.0, dtype=torch.float32).permute(2, 0, 1)


def _maps(image, depth):
    """Passa la imatge per VGG19 i retorna els mapes de contingut i els d'estil de les primeres depth capes."""
    x = (image - MEAN.to(image.device)) / STD.to(image.device)
    content, styles = None, []
    last = max(CONTENT_LAYER, STYLE_LAYERS[depth - 1])
    for position, layer in enumerate(_vgg()):
        x = layer(x)
        if position in STYLE_LAYERS[:depth]:
            styles.append(x)
        if position == CONTENT_LAYER:
            content = x
        if position == last:
            break
    return content, styles


def _texture(maps):
    """Resumeix uns mapes en quins filtres s'encenen junts, sense dir on: és la «textura» (matriu de Gram)."""
    _, channels, height, width = maps.shape
    flat = maps.reshape(channels, height * width)
    return flat @ flat.T / (channels * height * width)


def stylize(photo, painting, depth=3, style_weight=1e7, steps=None, size=None):
    """Pinta la foto amb l'estil del quadre i retorna la imatge resultant, un tensor (3, alt, ample).

    Paràmetres:
        photo, painting: rutes de la foto i del quadre.
        depth: de quantes capes de VGG19 es copia l'estil, d'1 (només colors) a 5 (pinzellades i formes).
        style_weight: quant pesa l'estil davant del contingut de la foto.
        steps: quantes passes de gradient descent es fan sobre els píxels.
        size: costat més llarg de la imatge, en píxels.
    Si no dius steps ni size, s'adapten a si tens GPU o no.
    """
    if depth not in (1, 2, 3, 4, 5):
        raise ValueError(f"depth ha de ser un enter d'1 a 5, les capes de VGG19 d'on es copia l'estil: {depth} no val.")
    default_size, default_steps = default_settings(DEVICE)
    size, steps = size or default_size, steps or default_steps
    content_image = load_picture(photo, size).unsqueeze(0).to(DEVICE)
    style_image = load_picture(painting, size).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        target_content, _ = _maps(content_image, depth)
        target_textures = [_texture(maps) for maps in _maps(style_image, depth)[1]]

    torch.manual_seed(SEED)
    image = content_image.clone().requires_grad_(True)
    step_maker = torch.optim.Adam([image], lr=0.03)
    for _ in range(steps):
        step_maker.zero_grad()
        content, styles = _maps(image, depth)
        style_loss = sum(nn.functional.mse_loss(_texture(maps), target)
                         for maps, target in zip(styles, target_textures)) / depth
        noise = ((image[..., 1:, :] - image[..., :-1, :]).abs().mean()
                 + (image[..., :, 1:] - image[..., :, :-1]).abs().mean())
        loss = nn.functional.mse_loss(content, target_content) + style_weight * style_loss + SMOOTHING * noise
        loss.backward()
        step_maker.step()
        with torch.no_grad():
            image.clamp_(0, 1)
    return image.detach()[0].cpu()


def as_picture(image):
    """Converteix un tensor (3, alt, ample) en una imatge de PIL, la que accepta predict de YOLO."""
    return Image.fromarray((image.clamp(0, 1).permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))


def show_style(photo, painting, result):
    """Dibuixa la foto, el quadre i el resultat, l'un al costat de l'altre."""
    fig, grafics = plt.subplots(1, 3, figsize=(12, 4))
    images = [load_picture(photo, 512), load_picture(painting, 512), result]
    for grafic, image, title in zip(grafics, images, ["foto", "quadre", "resultat"]):
        grafic.imshow(image.permute(1, 2, 0).numpy())
        grafic.set_title(title)
        grafic.axis("off")
    plt.tight_layout()
    plt.show()

Baixem quatre quadres de domini públic: *La nit estelada* de Van Gogh, *La gran onada de Kanagawa* de Hokusai, *El
crit* de Munch i *Composició VII* de Kandinski. I pintem la gata amb l'onada.

In [ ]:
PAINTINGS = ["nit_estelada.jpg", "onada.jpg", "crit.jpg", "kandinsky.jpg"]
for name in PAINTINGS:
    urllib.request.urlretrieve(URL_UT + "estils/" + name, name)

painted = stylize("gata.png", "onada.jpg")
show_style("gata.png", "onada.jpg", painted)

La gata conserva les formes, els ulls, el nas i les orelles, però agafa els colors i les textures de l'onada. **No
s'ha entrenat cap xarxa:** s'ha retocat la foto fins que VGG19 hi veu la gata i, alhora, la textura de Hokusai.

### Ara tu · Missió 5: el teu quadre

1. **Aposta.** Què creus que copiarà del quadre amb `depth=1`? I amb `depth=5`?
2. **Experimenta.** Pinta la mateixa foto amb `depth` = 1, 3 i 5, i després amb els quatre quadres. Prova també
   `style_weight` = 1e5, 1e7 i 1e9. Quina combinació t'agrada més?
3. **Foto pròpia (opcional).** Puja la foto d'un objecte amb la icona de la carpeta de Colab i pinta-la. **Només
   objectes, sense cares, persones ni dades privades.**
4. **Crash test.** Passa la gata pintada pel detector de la secció 1:
   `detector.predict(as_picture(painted), conf=0.25, verbose=False)[0]`. Encara hi veu un gat? Amb quin `depth` o
   quin `style_weight` deixa de veure'l?
5. **Explica.** Per què les primeres capes copien sobretot colors, i les profundes, formes més grans? Què et diu el
   punt 4 sobre on es fixa una CNN, en la forma o en la textura?

In [ ]:
# El teu codi aquí

## 6. Extra: una xarxa que llegeix

Fins ara, cada model tenia una llista tancada de classes: les 1.000 d'ImageNet al NB 6.4 i les 80 de COCO aquí.
**CLIP** (OpenAI, 2021) no en té cap: **les classes les escrius tu, amb text**, i el model tria la que s'assembla més
a la foto. Això es diu classificar **zero-shot**: sense cap foto d'exemple de la classe i sense entrenar res.

> **Incís: com ho fa?** CLIP són dues xarxes. Una mira la imatge i la converteix en una llista de 512 números, com les
> característiques de ResNet18 al NB 6.4. L'altra fa el mateix amb una frase. Es van entrenar juntes amb uns 400
> milions de parells *foto + peu de foto* trets d'internet, perquè una foto i la seva descripció acabin amb números
> semblants. Per classificar, CLIP compara la foto amb cada frase i una softmax converteix les semblances en
> percentatges que sumen 100. La part que mira la imatge d'aquest CLIP no és una CNN, sinó un *transformer* de visió,
> una família més recent; els autors també en van publicar versions amb ResNet, i s'usen exactament igual.

> **Norma.** OpenAI va publicar CLIP per a la recerca i n'exclou expressament la vigilància i el reconeixement
> facial. En aquesta missió, **no li facis preguntes sobre persones** (edat, gènere, origen, estat d'ànim…): les
> respondria amb la mateixa seguretat que tot el demés, i amb tots els biaixos de les fotos d'internet.

El model es baixa amb la biblioteca **transformers** de Hugging Face, que Colab ja porta instal·lada. La funció
`ask_clip` posa cada etiqueta dins d'una frase (el paràmetre `template`, per defecte `"a photo of {}."`) i retorna
el percentatge de cada una, de més a menys. Accepta el nom d'un fitxer o una imatge de PIL, com `predict` de YOLO.

> ⏱ La primera vegada baixa uns 600 MB: a Colab, menys d'un minut. Després, cada pregunta tarda dècimes de segon.

In [ ]:
!pip install -q transformers
from transformers import pipeline

clip = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")


def ask_clip(image, labels, template="a photo of {}."):
    """Retorna, en %, quant s'assembla la imatge a cada etiqueta, de més a menys."""
    answers = clip(image, candidate_labels=labels, hypothesis_template=template)
    return pd.Series({answer["label"]: round(100 * answer["score"], 1) for answer in answers}, name="%")


ask_clip("gata.png", ["cat", "dog", "rocket"])

**92,6 % gat**, i ningú no li ha ensenyat mai que «cat» era una de les seves classes: només ha llegit la paraula.
Les etiquetes van en anglès, perquè CLIP va aprendre amb peus de foto d'internet. Què passa en altres idiomes, ho
descobriràs tu.

### Ara tu · Missió extra: tria tu les classes

Si has acabat les altres missions, aquí tens CLIP per jugar-hi.

1. **Aposta.** A la Missió 2, el detector no sabia què era `coet.jpg`. Si dones a CLIP les etiquetes `rocket`,
   `lighthouse`, `pencil`, `chimney` i `candle`, què dirà? I si les escrius en català: `coet`, `far`, `llapis`,
   `xemeneia` i `espelma`?
2. **Experimenta.**
   - Prova les dues llistes amb el coet. Amb la catalana, prova també `template="una foto d'un {}."`.
   - **Fes-li una trampa:** passa-li la gata amb etiquetes que no hi tinguin res a veure. Què respon?
   - **Fes-li preguntes que YOLO no podia respondre:** en quina estació de l'any és `carrer_neu.jpg`? De quina
     època és `dublin_1901.jpg`? Quin cafè hi ha a `cafe.png`? Prova `template="a photo taken in {}."` o la frase
     que se t'acudeixi.
   - Compara la frase per defecte amb `template="{}"`, que passa l'etiqueta tota sola.
3. **Explica.** Per què funciona tan diferent en català? Què fa quan cap etiqueta és bona, i què et recorda de
   ResNet18 al NB 6.4? Quin avantatge té escriure tu les classes en lloc de tenir-ne 80 de fixes? I quin perill?

In [ ]:
# El teu codi aquí

## 7. Resum

| | Classificar | Detectar | Segmentar | Poses |
|---|---|---|---|---|
| **Quants objectes** | Un per imatge | Molts | Molts | Moltes persones |
| **Què en sap** | La classe | Classe i requadre | Classe i píxels | Persona i 17 punts |
| **Per a què** | Ordenar fotos | Comptar, vigilar | Editar, mesurar | Esport, gestos, animació |

### Les idees que t'has d'endur

1. **El llindar és una decisió de negoci,** no un detall tècnic: tria entre avisar de més o avisar de menys.
2. **Un model no surt de les seves classes.** Si no és a la llista, no ho veurà o s'ho inventarà.
3. **Una CNN continua sent una CNN:** per més impressionant que sigui, aguanta el que ha vist.
4. **Ensenyar-li coses noves és transfer learning,** i el que no li tornes a ensenyar, ho oblida.
5. **Una CNN separa el què del com:** les formes viuen a les capes profundes; els colors i les textures, a les primeres.
6. **Extra:** amb CLIP, les classes les escrius tu. Però només pot triar entre les que li dones, i només entén el que ha llegit.

## Exercici final

**Exercici 6. El comptador.** Escriu una funció `count_objects(path, class_name, conf=0.25)` que digui quants objectes
d'una classe hi ha en una foto, i prova-la amb les fotos de mostra. Compara-la amb el que comptes tu a ull. Amb quin llindar
s'acosta més? Per què no sempre n'hi ha prou amb una sola foto per triar-lo?

## Per al debat de classe

1. **Vigilància.** Tot el que heu fet avui es pot fer amb una càmera de carrer i un portàtil. Hi ha ciutats que ho
   fan servir per comptar gent, o per identificar-la. On posaries la línia? A Europa, el Reglament d'IA limita alguns
   d'aquests usos.
2. **Consentiment.** Les persones de Dublín de 1901 no van autoritzar res, i les de les fotos amb què s'han entrenat
   aquests models, tampoc gaire. Passa alguna cosa?
3. **Errors desiguals.** Si un detector de persones falla més amb determinats colors de pell o de roba, qui hauria de
   comprovar-ho abans de vendre'l?
4. **Qui escriu les etiquetes.** CLIP classifica en qualsevol categoria que li escriguis, també en les que no hauria de fer servir ningú. Qui n'és responsable: qui fa el model o qui escriu les etiquetes?